# Phase 4 · Data analysis — Court Judgment Prediction (CJPE)

Computes the statistics needed to design the Phase 4 models and writes two small files to send back:
- `P4_analysis/p4_stats.json` — sizes, label balance, character / word / BPE-token length distributions, chunk counts, overlaps between splits, keyword statistics near the end of documents.
- `P4_analysis/p4_samples.json` — a few example cases (beginning and end of the text only) for each label.

**Kaggle setup:** CPU is enough (no GPU needed) · Add-ons → Secrets: **`HF`** · Internet on · Add Input → **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer). Expected time ≈ 10–15 minutes (tokenizing every document dominates).

In [ ]:
# Cell 1 — Imports and output folder.
import json, glob, random, hashlib, re, statistics as st
from pathlib import Path
from collections import Counter
import numpy as np
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

DATA_DIR = Path("/kaggle/working/data")
OUT_DIR = Path("/kaggle/working/P4_analysis")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)
print("Outputs:", OUT_DIR)

In [ ]:
# Cell 2 — Download Phase_4_CJPE from Hugging Face using the Kaggle secret "HF".
snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                  token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_4_CJPE/**"])
P4_DIR = DATA_DIR / "Phase_4_CJPE"
for f in sorted(P4_DIR.rglob("*.json")):
    print(f"{f.relative_to(P4_DIR)}: {f.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Cell 3 — Load the three splits and show their record structure (field names and types).
def load(path):
    data = json.load(open(path))
    return data if isinstance(data, list) else [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v}
                                                 for k, v in data.items()]

splits = {"train": load(P4_DIR / "Train/train.json"), "dev": load(P4_DIR / "Train/dev.json"),
          "test": load(P4_DIR / "Test/test.json")}
stats = {"files": {}, "fields": {}}
for name, docs in splits.items():
    first = docs[0]
    stats["fields"][name] = {k: type(v).__name__ for k, v in first.items()}
    print(f"{name}: {len(docs)} cases | fields: {stats['fields'][name]}")
    print("   example:", {k: (v[:80] + "…" if isinstance(v, str) and len(v) > 80 else v) for k, v in first.items()})

In [ ]:
# Cell 4 — Label field and label balance per split (Macro-F1 depends on both classes).
LABEL_KEY = next(k for k in ("label", "labels", "pred", "outcome", "judgment") if k in splits["train"][0])
TEXT_KEY = "text"
stats["label_key"] = LABEL_KEY
stats["label_counts"] = {}
for name in ("train", "dev"):
    counts = Counter(int(d[LABEL_KEY]) for d in splits[name])
    stats["label_counts"][name] = dict(counts)
    total = sum(counts.values())
    print(f"{name}: " + ", ".join(f"label {k}: {v} ({100 * v / total:.1f}%)" for k, v in sorted(counts.items())))
stats["n_cases"] = {name: len(docs) for name, docs in splits.items()}

In [ ]:
# Cell 5 — Character and word lengths per split (percentiles), and empty / very short documents.
def pct(xs, ps=(0, 10, 25, 50, 75, 90, 95, 99, 100)):
    arr = np.array(xs)
    return {f"p{p}": int(np.percentile(arr, p)) for p in ps} | {"mean": int(arr.mean())}

stats["chars"], stats["words"], stats["short_docs"] = {}, {}, {}
for name, docs in splits.items():
    chars = [len(d[TEXT_KEY]) for d in docs]
    words = [len(d[TEXT_KEY].split()) for d in docs]
    stats["chars"][name], stats["words"][name] = pct(chars), pct(words)
    stats["short_docs"][name] = sum(w < 50 for w in words)
    print(f"{name}: words {stats['words'][name]} | docs under 50 words: {stats['short_docs'][name]}")

In [ ]:
# Cell 6 — BPE-token lengths with the Phase-1 tokenizer, and how many 512-token chunks each case needs.
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
stats["tokens"], stats["chunks_512"], stats["tokens_per_word"] = {}, {}, {}
token_lengths = {}
for name, docs in splits.items():
    lengths = []
    for start in range(0, len(docs), 256):
        lengths += [len(e.ids) for e in tokenizer.encode_batch([d[TEXT_KEY] for d in docs[start:start + 256]])]
    token_lengths[name] = lengths
    stats["tokens"][name] = pct(lengths)
    stats["chunks_512"][name] = pct([max(1, -(-n // 510)) for n in lengths])
    stats["tokens_per_word"][name] = round(sum(lengths) / max(1, sum(len(d[TEXT_KEY].split()) for d in docs)), 3)
    print(f"{name}: tokens {stats['tokens'][name]}")
    print(f"      512-token chunks per case {stats['chunks_512'][name]} | total tokens {sum(lengths):,}")
for name in ("train", "dev"):
    by_label = {}
    for d, n in zip(splits[name], token_lengths[name]):
        by_label.setdefault(int(d[LABEL_KEY]), []).append(n)
    stats.setdefault("tokens_by_label", {})[name] = {str(k): pct(v, (25, 50, 75)) for k, v in sorted(by_label.items())}
    print(f"{name} token length by label: {stats['tokens_by_label'][name]}")

In [ ]:
# Cell 7 — Overlaps between splits: shared ids and exact duplicate texts (leakage check).
def text_hash(t):
    return hashlib.md5(" ".join(t.split()).encode()).hexdigest()

ids = {name: {str(d["id"]) for d in docs} for name, docs in splits.items()}
hashes = {name: Counter(text_hash(d[TEXT_KEY]) for d in docs) for name, docs in splits.items()}
stats["overlap"] = {
    "ids_train_dev": len(ids["train"] & ids["dev"]), "ids_train_test": len(ids["train"] & ids["test"]),
    "ids_dev_test": len(ids["dev"] & ids["test"]),
    "text_train_dev": len(set(hashes["train"]) & set(hashes["dev"])),
    "text_train_test": len(set(hashes["train"]) & set(hashes["test"])),
    "text_dev_test": len(set(hashes["dev"]) & set(hashes["test"])),
    "duplicate_texts_within_train": sum(c - 1 for c in hashes["train"].values() if c > 1),
}
# Do duplicated training texts always carry the same label?
by_hash = {}
for d in splits["train"]:
    by_hash.setdefault(text_hash(d[TEXT_KEY]), set()).add(int(d[LABEL_KEY]))
stats["overlap"]["train_duplicates_with_conflicting_labels"] = sum(len(v) > 1 for v in by_hash.values())
print(json.dumps(stats["overlap"], indent=2))

In [ ]:
# Cell 8 — Where is the outcome signal? Keyword rates in the first vs last 1,000 characters, per label.
KEYWORDS = ["allowed", "dismissed", "set aside", "quashed", "upheld", "affirmed", "remanded", "disposed",
            "appeal", "petition", "conviction", "acquitted", "costs", "accordingly", "hereby"]
stats["keywords"] = {}
for name in ("train", "dev"):
    table = {}
    for part in ("first_1000", "last_1000"):
        for label in sorted({int(d[LABEL_KEY]) for d in splits[name]}):
            docs = [d for d in splits[name] if int(d[LABEL_KEY]) == label]
            snippets = [(d[TEXT_KEY][:1000] if part == "first_1000" else d[TEXT_KEY][-1000:]).lower() for d in docs]
            table[f"{part}_label{label}"] = {k: round(sum(k in s for s in snippets) / len(snippets), 3) for k in KEYWORDS}
    stats["keywords"][name] = table
for key, row in stats["keywords"]["train"].items():
    print(key, {k: v for k, v in row.items() if v > 0.05})

In [ ]:
# Cell 9 — Section markers: how often typical judgment headings appear (helps decide section selection).
MARKERS = {"facts": r"\bfacts?\b", "arguments": r"\b(learned counsel|contended|submitted that|argued)\b",
           "issue": r"\b(question|issue)s? (for|that arise|arising)\b", "held": r"\bheld\b",
           "order": r"\border\b", "judgment": r"\bjudgment\b", "section": r"\bsection \d+",
           "article": r"\barticle \d+"}
sample = random.sample(splits["train"], min(2000, len(splits["train"])))
stats["section_markers_train_sample"] = {k: round(sum(bool(re.search(p, d[TEXT_KEY], re.I)) for d in sample) / len(sample), 3)
                                         for k, p in MARKERS.items()}
print(stats["section_markers_train_sample"])

In [ ]:
# Cell 10 — Save the statistics and a few example cases (first and last 1,500 characters only).
samples = {}
for name in ("train", "dev"):
    for label in sorted({int(d[LABEL_KEY]) for d in splits[name]}):
        chosen = random.sample([d for d in splits[name] if int(d[LABEL_KEY]) == label], 3)
        samples[f"{name}_label{label}"] = [{"id": str(d["id"]), "n_chars": len(d[TEXT_KEY]),
                                           "start": d[TEXT_KEY][:1500], "end": d[TEXT_KEY][-1500:]} for d in chosen]
samples["test"] = [{"id": str(d["id"]), "n_chars": len(d[TEXT_KEY]), "start": d[TEXT_KEY][:1500],
                    "end": d[TEXT_KEY][-1500:]} for d in random.sample(splits["test"], 3)]
json.dump(stats, open(OUT_DIR / "p4_stats.json", "w"), indent=2)
json.dump(samples, open(OUT_DIR / "p4_samples.json", "w"), indent=2)
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e3:.1f} KB")
print("Download both files from the Output panel (P4_analysis/) and send them.")